In [ ]:
import os
import pandas as pd
import numpy as np

# 导入工具：规范类型 / 索引 / 按日覆盖（与后端 config、看板查询对齐）
from db_export import DataFrameToMySQLWithMetadata, export_brand_tables, normalize_dataframe, begin_import_batch


In [ ]:
# Load 所有门店表
fpath1 = "../data/品牌2门店表.xlsx"
xj_info = pd.read_excel(fpath1, sheet_name='门店名称')

fpath2 = "../data/品牌3门店表.xlsx"
zz_info = pd.read_excel(fpath2, sheet_name='门店名称')

fpath4 = "../data/品牌1门店表.xlsx"
kp_info = pd.read_excel(fpath4, sheet_name='门店名称')


In [ ]:
# 建表与写入逻辑已抽到 db_export.py
# - 日字段 DATE、金额 DECIMAL、门店/订单 ID VARCHAR
# - 复合索引 (日期/下单时间, 平台, 门店id) 服务看板过滤
# - append 时按日期区间先删后插，避免重复导入翻倍
# - 连接信息优先读环境变量 MYSQL_HOST / MYSQL_USER / MYSQL_PASSWORD / MYSQL_DATABASE / MYSQL_PORT

import logging
logging.basicConfig(level=logging.INFO)

def make_exporter():
    return DataFrameToMySQLWithMetadata(
        host=os.getenv("MYSQL_HOST", "127.0.0.1"),
        user=os.getenv("MYSQL_USER", "root"),
        password=os.getenv("MYSQL_PASSWORD", ""),  # 勿把密码写进 notebook，用环境变量
        database=os.getenv("MYSQL_DATABASE", "fastapi_db"),
        port=int(os.getenv("MYSQL_PORT", "3306")),
    )


In [ ]:
# 品牌2 商品数据处理
# 美团商品数据表
fpath1 = "../data/美团订单数据.csv"
data1 = pd.read_csv(fpath1, encoding = 'gbk')
data1['下单时间'] = pd.to_datetime(data1['下单时间'], errors='coerce').dt.normalize()
data1 = data1[data1['订单状态']!='订单取消']
# 去除表格所有字符串后面的空格
obj_cols = data1.select_dtypes(include=['object']).columns
data1[obj_cols] = data1[obj_cols].apply(lambda s: s.str.rstrip())
data1['商品销售额'] = data1['商品原价销售额']
data1 = data1[['店铺ID','下单时间','商品分类', '商品名称', '商品销售数量', '商品销售额']]
data1['平台'] = '美团'
# 连接门店信息表
xj_mt_product = pd.merge(data1, xj_info[['机构编码', '战区', '门店名称', '美团门店ID']], how='left', left_on='店铺ID', right_on='美团门店ID')
xj_mt_product = xj_mt_product.drop(columns=['美团门店ID'])

# 饿了么商品数据表
fpath3 = '../data/品牌2饿了么订单.xlsx'
fpath7 = '../data/品牌2饿了么订单-门店示例.xlsx'
data3 = pd.read_excel(fpath3, sheet_name='订单导出')
data7 = pd.read_excel(fpath7, sheet_name='订单导出')
data3 = pd.concat([data3, data7], ignore_index=True)
data3['订单状态'] = data3['订单状态'].ffill()
data3['下单时间'] = data3['下单时间'].ffill()
data3 = data3[data3['订单状态'] == '已完结']
# 替换值
data3['下单时间'] = data3['下单时间'].str.replace('年', '-').str.replace('月', '-').str.replace('日', '')
data3['下单时间'] = pd.to_datetime(data3['下单时间'], errors='coerce').dt.normalize()
data3['商品销售额'] = data3['优惠前单价'] * data3['购买数量']
data3['商品分类'] = np.where(data3['商品分类'].isnull(), '人气必买', data3['商品分类'])
data3 = data3[['门店ID','下单时间','商品分类', '商品名称', '购买数量', '商品销售额']]
data3['平台'] = '饿了么'
# 连接门店信息表
xj_elm_product = pd.merge(data3, xj_info[['机构编码', '战区', '门店名称', '饿了么门店ID']], how='left', left_on='门店ID', right_on='饿了么门店ID')
xj_elm_product = xj_elm_product.drop(columns=['饿了么门店ID'])
xj_elm_product = xj_elm_product.rename(columns={'门店ID':'店铺ID', '购买数量':'商品销售数量'})

xj_product = pd.concat([xj_mt_product, xj_elm_product], ignore_index=True)


In [ ]:
# 品牌2 订单数据处理
# 美团订单数据表
fpath2 = "../data/美团订单数据.csv"
data2 = pd.read_csv(fpath2, encoding = 'gbk')
time_column = data2['下单时间'].values
data2['下单时间'] = np.array(time_column, dtype='datetime64')
data2['订单金额'] = data2['订单总金额'] - data2['配送费']
data2['订单实付'] = data2['订单折扣后金额']
data2 = data2[data2['订单状态']!='订单取消']
data2 = data2[['订单编号','店铺ID','下单时间','订单金额', '订单实付']]
# 去重
data2 = data2.drop_duplicates(subset=['订单编号'])
data2['平台'] = '美团'
# 按照下单时间进行分类
# 零售时间段划分：上午（00:00-12:00），下午（12:00-18:00），晚上（18:00-24:00）
data2['下单时间分类'] = np.where((data2['下单时间'].dt.hour >= 0) & (data2['下单时间'].dt.hour < 12), '上午',np.where((data2['下单时间'].dt.hour >= 12) & (data2['下单时间'].dt.hour < 18), '下午',np.where((data2['下单时间'].dt.hour >= 18) & (data2['下单时间'].dt.hour < 24), '晚上', '其他')))
# 按订单金额进行分类
data2['下单金额分类'] = np.where((data2['订单金额'] >= 30) & (data2['订单金额'] < 38), '30-38元',np.where((data2['订单金额'] >= 38) & (data2['订单金额'] < 49), '38-49元',np.where((data2['订单金额'] >= 49) & (data2['订单金额'] < 69), '49-69元',np.where((data2['订单金额'] >= 69) & (data2['订单金额'] < 89), '69-89元',np.where((data2['订单金额'] >= 89) & (data2['订单金额'] < 120), '89-120元',np.where((data2['订单金额'] >= 120), '120元以上', '其他'))))))
# 按订单实付进行分类
data2['订单实付分类'] = np.where(data2['订单实付'] < 30,'0-30元', np.where((data2['订单实付'] >= 30) & (data2['订单实付'] < 38), '30-38元',np.where((data2['订单实付'] >= 38) & (data2['订单实付'] < 49), '38-49元',np.where((data2['订单实付'] >= 49) & (data2['订单实付'] < 69), '49-69元',np.where((data2['订单实付'] >= 69) & (data2['订单实付'] < 89), '69-89元',np.where((data2['订单实付'] >= 89) & (data2['订单实付'] < 120), '89-120元',np.where((data2['订单实付'] >= 120), '120元以上', '其他')))))))
# 连接门店信息表
xj_mt_order = pd.merge(data2, xj_info[['机构编码', '战区', '门店名称', '美团门店ID']], how='left', left_on='店铺ID', right_on='美团门店ID')
xj_mt_order = xj_mt_order.drop(columns=['美团门店ID'])

# 饿了么订单数据表
fpath3 = '../data/品牌2饿了么订单.xlsx'
fpath7 = '../data/品牌2饿了么订单-门店示例.xlsx'
data3 = pd.read_excel(fpath3, sheet_name='订单导出')
data7 = pd.read_excel(fpath7, sheet_name='订单导出')
data3 = pd.concat([data3, data7], ignore_index=True)
data3 = data3[data3['订单状态'] =='已完结']
# 去重
data3 = data3.drop_duplicates(subset=['订单编号'])
# 替换值
data3['下单时间'] = data3['下单时间'].str.replace('年', '-').str.replace('月', '-').str.replace('日', '')
data3['订单金额'] = data3['订单总金额'] - data3['配送费']
data3['订单实付'] = data3['用户实付金额']
time_column = data3['下单时间'].values
data3['下单时间'] = np.array(time_column, dtype='datetime64')
data3['下单时间分类'] = np.where((data3['下单时间'].dt.hour >= 0) & (data3['下单时间'].dt.hour < 12), '上午',np.where((data3['下单时间'].dt.hour >= 12) & (data3['下单时间'].dt.hour < 18), '下午',np.where((data3['下单时间'].dt.hour >= 18) & (data3['下单时间'].dt.hour < 24), '晚上', '其他')))
# 按订单金额进行分类
data3['下单金额分类'] = np.where((data3['订单金额'] >= 30) & (data3['订单金额'] < 38), '30-38元',np.where((data3['订单金额'] >= 38) & (data3['订单金额'] < 49), '38-49元',np.where((data3['订单金额'] >= 49) & (data3['订单金额'] < 69), '49-69元',np.where((data3['订单金额'] >= 69) & (data3['订单金额'] < 89), '69-89元',np.where((data3['订单金额'] >= 89) & (data3['订单金额'] < 120), '89-120元',np.where((data3['订单金额'] >= 120), '120元以上', '其他'))))))
# 按订单实付进行分类
data3['订单实付分类'] = np.where(data3['订单实付'] < 30,'0-30元',np.where((data3['订单实付'] >= 30) & (data3['订单实付'] < 38), '30-38元',np.where((data3['订单实付'] >= 38) & (data3['订单实付'] < 49), '38-49元',np.where((data3['订单实付'] >= 49) & (data3['订单实付'] < 69), '49-69元',np.where((data3['订单实付'] >= 69) & (data3['订单实付'] < 89), '69-89元',np.where((data3['订单实付'] >= 89) & (data3['订单实付'] < 120), '89-120元',np.where((data3['订单实付'] >= 120), '120元以上', '其他')))))))
# 连接门店信息表
xj_elm_order = data3[['订单编号','门店ID','下单时间','订单金额','订单实付','下单时间分类', '下单金额分类', '订单实付分类']]
xj_elm_order = pd.merge(xj_elm_order, xj_info[['机构编码', '战区', '门店名称', '饿了么门店ID']], how='left', left_on='门店ID', right_on='饿了么门店ID')
xj_elm_order = xj_elm_order.drop(columns=['饿了么门店ID'])
xj_elm_order = xj_elm_order.rename(columns={'门店ID':'店铺ID'})
xj_elm_order['平台'] = '饿了么'

# 京东订单数据表
fpath6 = "../data/京东账单.xlsx"
data6 = pd.read_excel(fpath6, skiprows=1)
data6 = data6[data6['订单类型']=='正向订单']
data6['订单金额'] = data6['订单原价'] + data6['餐盒费']
data6['订单实付'] = data6['用户支付货款']
time_column = data6['订单完成/退款完成时间'].values
data6['下单时间'] = np.array(time_column, dtype='datetime64')
data6['下单时间分类'] = np.where((data6['下单时间'].dt.hour >= 0) & (data6['下单时间'].dt.hour < 12), '上午',np.where((data6['下单时间'].dt.hour >= 12) & (data6['下单时间'].dt.hour < 18), '下午',np.where((data6['下单时间'].dt.hour >= 18) & (data6['下单时间'].dt.hour < 24), '晚上', '其他')))
# 按订单金额进行分类
data6['下单金额分类'] = np.where((data6['订单金额'] >= 30) & (data6['订单金额'] < 38), '30-38元',np.where((data6['订单金额'] >= 38) & (data6['订单金额'] < 49), '38-49元',np.where((data6['订单金额'] >= 49) & (data6['订单金额'] < 69), '49-69元',np.where((data6['订单金额'] >= 69) & (data6['订单金额'] < 89), '69-89元',np.where((data6['订单金额'] >= 89) & (data6['订单金额'] < 120), '89-120元',np.where((data6['订单金额'] >= 120), '120元以上', '其他'))))))
# 按订单实付进行分类
data6['订单实付分类'] =np.where(data6['订单实付'] < 30, '0-30元', np.where((data6['订单实付'] >= 30) & (data6['订单实付'] < 38), '30-38元',np.where((data6['订单实付'] >= 38) & (data6['订单实付'] < 49), '38-49元',np.where((data6['订单实付'] >= 49) & (data6['订单实付'] < 69), '49-69元',np.where((data6['订单实付'] >= 69) & (data6['订单实付'] < 89), '69-89元',np.where((data6['订单实付'] >= 89) & (data6['订单实付'] < 120), '89-120元',np.where((data6['订单实付'] >= 120), '120元以上', '其他')))))))
# 连接门店信息表
xj_jd_order = data6[['主订单号','门店编号','下单时间','订单金额','订单实付','下单时间分类', '下单金额分类', '订单实付分类']]
xj_jd_order = pd.merge(xj_jd_order, xj_info[['机构编码', '战区', '门店名称', '京东门店ID']], how='left', left_on='门店编号', right_on='京东门店ID')
xj_jd_order = xj_jd_order.drop(columns=['京东门店ID'])
xj_jd_order = xj_jd_order.rename(columns={'门店编号':'店铺ID', '主订单号':'订单编号'})
xj_jd_order['平台'] = '京东'

xj_order = pd.concat([xj_mt_order, xj_elm_order, xj_jd_order], ignore_index=True)
# 保留完整下单时间（时段已写入分类列）；勿 normalize，便于 DATETIME 索引范围查询


In [ ]:
# 品牌2 流量数据处理
# 美团流量数据表
fpath1 = "../data/美团流量数据.csv"
data1 = pd.read_csv(fpath1, encoding = 'gbk', parse_dates=['日期'])
time_column = data1['日期'].values
data1['日期'] = np.array(time_column, dtype='datetime64[ns]')
data1 = data1[['日期','门店ID','曝光人数','入店人数','下单人数']]
xj_mt_exposure = pd.merge(data1, xj_info[['机构编码', '战区', '门店名称', '美团门店ID']], how='left', left_on='门店ID', right_on='美团门店ID')
xj_mt_exposure['平台'] = '美团'
xj_mt_exposure = xj_mt_exposure.rename(columns={'开始时间':'日期','门店ID':'门店id'})
xj_mt_exposure = xj_mt_exposure.drop(columns=['美团门店ID'])

# 饿了么流量数据表
fpath2 = "../data/饿了么区域示例-流量.xlsx"
fpath7 = "../data/品牌2饿了么流量-门店示例.xlsx"
data2 = pd.read_excel(fpath2, sheet_name='data', parse_dates=['日期'])
data7 = pd.read_excel(fpath7, sheet_name='data', parse_dates=['日期'])
data2 = pd.concat([data2, data7], ignore_index=True)
time_column = data2['日期'].values
data2['日期'] = np.array(time_column, dtype='datetime64[ns]')
data2 = data2[['日期','门店id','总曝光人数','总进店人数','总下单人数']]
data2 = data2.rename(columns={'总曝光人数':'曝光人数','总进店人数':'入店人数','总下单人数':'下单人数'})
xj_elm_exposure = pd.merge(data2, xj_info[['机构编码', '战区', '门店名称', '饿了么门店ID']], how='left', left_on='门店id', right_on='饿了么门店ID')
xj_elm_exposure['平台'] = '饿了么'
xj_elm_exposure = xj_elm_exposure.drop(columns=['饿了么门店ID'])

# 京东流量数据表
fpath3 = "../data/京东流量数据.xlsx"
data3 = pd.read_excel(fpath3, parse_dates=['日期'])
time_column = data3['日期'].values
data3['日期'] = np.array(time_column, dtype='datetime64[ns]')
data3 = data3[['日期','门店ID','曝光人数','入店人数','下单人数']]
xj_jd_exposure = pd.merge(data3, xj_info[['机构编码', '战区', '门店名称', '京东门店ID']], how='left', left_on='门店ID', right_on='京东门店ID')
xj_jd_exposure['平台'] = '京东'
xj_jd_exposure = xj_jd_exposure.rename(columns={'门店ID':'门店id'})
xj_jd_exposure = xj_jd_exposure.drop(columns=['京东门店ID'])

# 合并数据
xj_exposure = pd.concat([xj_mt_exposure, xj_elm_exposure, xj_jd_exposure], ignore_index=True)


In [ ]:
# 品牌2 财务数据处理
# 美团财务数据表
fpath1 = "../data/美团财务数据.csv"
data1 = pd.read_csv(fpath1, encoding = 'gbk', parse_dates=['开始时间'])
time_column = data1['开始时间'].values
data1['开始时间'] = np.array(time_column, dtype='datetime64[D]')
data1['营业额'] = data1['包装费'] + data1['商品原价']
data1['平台佣金'] = data1['佣金']+ data1['配送服务费']
data1['平台'] = '美团'
data1_account = data1[['平台','开始时间','商家ID','营业额','有效订单数','平台佣金','商家补贴金额','收入']]
xj_mt_account = data1_account.rename(columns={'开始时间':'日期','商家ID':'门店id'})
xj_mt_account = pd.merge(xj_mt_account, xj_info[['机构编码', '战区', '门店名称', '美团门店ID']], how='left', left_on='门店id', right_on='美团门店ID')
xj_mt_account = xj_mt_account.drop(columns=['美团门店ID'])
xj_mt_account['有效订单数'] = np.where(xj_mt_account['有效订单数']<0, 0, xj_mt_account['有效订单数'])

#饿了么财务数据表
fpath3 = "../data/饿了么区域示例.xlsx"
fpath7 = "../data/品牌2饿了么财务-门店示例.xlsx"
data3 = pd.read_excel(fpath3, sheet_name='data', parse_dates=['日期'])
data7 = pd.read_excel(fpath7, sheet_name='data', parse_dates=['日期'])
data3 = pd.concat([data3, data7], ignore_index=True)
time_column = data3['日期'].values
data3['日期'] = np.array(time_column, dtype='datetime64[D]')
data3['营业额'] = data3['商品金额'] + data3['包装费']
data3['平台佣金'] = data3['营业额']- data3['预计收入（元）']- data3['商家补贴（元）']
data3['平台'] = '饿了么'
data3_account = data3[['平台','日期','门店id','营业额','有效订单','平台佣金','商家补贴（元）','预计收入（元）']]
xj_elm_account = data3_account.rename(columns={'商家补贴（元）':'商家补贴金额','预计收入（元）':'收入', '有效订单':'有效订单数'})
xj_elm_account = pd.merge(xj_elm_account, xj_info[['机构编码', '战区', '门店名称', '饿了么门店ID']], how='left', left_on='门店id', right_on='饿了么门店ID')
xj_elm_account = xj_elm_account.drop(columns=['饿了么门店ID'])


# 京东账单数据表
fpath6 = "../data/京东账单.xlsx"
data6 = pd.read_excel(fpath6, skiprows=1)
data6 = data6[data6['订单类型']=='正向订单']
data6['商家补贴金额'] = abs(data6['商家承担补贴(货款+运费)'])
data6['营业额'] = data6['订单原价'] + data6['餐盒费']
data6['收入'] = np.where(data6['订单类型']!='保险单', data6['应结金额'], 0)
data6['平台佣金'] = abs(data6['配送费（开票）'] + data6['总服务费(货款服务费+运费服务费+餐盒费服务费)'])
time_column = data6['订单完成/退款完成时间'].values
data6['日期'] = np.array(time_column, dtype='datetime64[D]')
data7 = data6.groupby(['日期','门店编号']).agg({'营业额':'sum','收入':'sum','门店名称':'count','平台佣金':'sum','商家补贴金额':'sum'})
data7 = data7.reset_index()
data7['平台'] = '京东'
data7_account = data7[['平台','日期','门店编号','营业额','门店名称','平台佣金','商家补贴金额','收入']]
xj_jd_account = data7_account.rename(columns={'门店编号':'门店id','门店名称':'有效订单数'})
xj_jd_account = pd.merge(xj_jd_account, xj_info[['机构编码', '战区', '门店名称', '京东门店ID']], how='left', left_on='门店id', right_on='京东门店ID')
xj_jd_account = xj_jd_account.drop(columns=['京东门店ID'])
xj_account = pd.concat([xj_mt_account,xj_elm_account, xj_jd_account], ignore_index=True)


In [ ]:
# 品牌1财务数据处理
# 美团数据表
fpath1 = "../data/品牌1美团数据.csv"
m_data = pd.read_csv(fpath1, encoding = 'gbk', parse_dates=['日期'])
time_column = m_data['日期'].values
m_data['日期'] = np.array(time_column, dtype='datetime64')
m_data[m_data.select_dtypes(include=['number']).columns] = m_data.select_dtypes(include=['number']).fillna(0)
m_data['营业额'] = m_data['商品原价'] + m_data['包装费']
m_data['平台佣金'] = m_data['平台服务费(含佣金和配送服务费)']
use_m_data1 = m_data[['日期','门店id','营业额','有效订单','平台佣金','商家活动支出','营业收入']]
kp_mt_account = use_m_data1.rename(columns={'有效订单':'有效订单数', '商家活动支出':'商家补贴金额', '营业收入':'收入'})
kp_mt_account['平台'] = '美团'
# 连接门店信息表
kp_mt_account = pd.merge(kp_mt_account, kp_info[['门店名称', '美团门店ID', '城市']], how='left', left_on='门店id', right_on='美团门店ID')
kp_mt_account = kp_mt_account.drop(columns=['美团门店ID'])

# 饿了么数据表
fpath2 = "../data/品牌1饿了么数据.xlsx"
e_data = pd.read_excel(fpath2, sheet_name='data', parse_dates=['日期'])
time_column = e_data['日期'].values
e_data['日期'] = np.array(time_column, dtype='datetime64')
e_data[e_data.select_dtypes(include=['number']).columns] = e_data.select_dtypes(include=['number']).fillna(0)
e_data['营业额'] = e_data['商品销售额'] + e_data['打包费']
e_data['平台佣金'] = e_data['平台技术服务费'] + e_data['履约技术服务费']
use_e_data1 = e_data[['日期','门店编号','营业额','有效订单','平台佣金','商家活动成本（含满减活动）','收入']]
kp_elm_account = use_e_data1.rename(columns={'门店编号':'门店id','有效订单':'有效订单数', '商家活动成本（含满减活动）':'商家补贴金额'})
kp_elm_account['平台'] = '饿了么'
kp_elm_account = pd.merge(kp_elm_account, kp_info[['门店名称', '饿了么门店ID', '城市']], how='left', left_on='门店id', right_on='饿了么门店ID')
kp_elm_account = kp_elm_account.drop(columns=['饿了么门店ID'])

# 合并数据
kp_account = pd.concat([kp_mt_account ,kp_elm_account], ignore_index=True)


In [ ]:
# 品牌1利润数据处理
# 导入品牌1菜单
fpath = "../data/品牌1菜单.xlsx"
p_data = pd.read_excel(fpath, sheet_name='Sheet1')
# 美团商品数据处理
fpath2 = '../data/品牌1美团财务数据.xlsx'
a_data = pd.read_excel(fpath2, sheet_name='订单明细', parse_dates=['账单日期'])
time_column = a_data['账单日期'].values
a_data['账单日期'] = np.array(time_column, dtype='datetime64')
a_data = a_data[a_data['交易类型'] == '外卖订单']
a_data['打包费'] = np.where(a_data['打包费'] == '-', 0, a_data['打包费'])
a_data = a_data[['门店id', '门店名称', '账单日期', '订单号', '商家应收款', '商品总价', '打包费']]
a_data['营业额'] = a_data['商品总价'].astype(float) + a_data['打包费'].astype(float)
a_sum_data = a_data.groupby('订单号')[['商家应收款', '营业额']].sum()
a_sum_data = a_sum_data.reset_index()
a_sum_data = a_sum_data.rename(columns={'订单号': '订单单号'})
account_data = pd.merge(a_sum_data, a_data[['门店id', '门店名称', '账单日期', '订单号']], how='left', left_on='订单单号', right_on='订单号')
account_data = account_data.drop_duplicates(subset=['订单单号'],keep='first')
account_data = account_data.drop(columns='订单号')
account_data['订单单号'] = account_data['订单单号'].astype(str)
account_data = account_data.rename(columns={"商家应收款": "结算金额"})

fpath = '../data/品牌1美团订单数据.csv'
m_data = pd.read_csv(fpath, encoding='gbk')
m_data = m_data[['订单编号', '商品信息']]
m_data['订单编号'] = m_data['订单编号'].str.split(":").str[1]
temp = m_data['商品信息'].str.split('/',expand=True)
actual_cols_num = temp.shape[1]
prefix = '商品名称'
dynamic_names_1 = [f"{prefix}{i+1}" for i in range(actual_cols_num)]
split_result_renamed1 = temp.rename(columns=dict(zip(temp.columns, dynamic_names_1)))
m_data = pd.concat([m_data.drop('商品信息', axis=1), split_result_renamed1], axis=1)
m_data = m_data.melt(
    id_vars=['订单编号'],
    var_name='商品名称',
    value_name='商品信息'
)
m_data = m_data.drop(columns=['商品名称'])

temp = m_data['商品信息'].str.split('数量',expand=True)
actual_cols_num = temp.shape[1]
prefix = '商品名称'
dynamic_names_1 = [f"{prefix}{i+1}" for i in range(actual_cols_num)]
split_result_renamed1 = temp.rename(columns=dict(zip(temp.columns, dynamic_names_1)))
m_data = pd.concat([m_data.drop('商品信息', axis=1), split_result_renamed1], axis=1)
m_data['销量'] = m_data.iloc[:, 2:].sum(axis=1)
m_data = m_data.rename(columns={'商品名称1':'商品名称'})
m_data = m_data[['订单编号', '商品名称', '销量']]

temp = m_data['商品名称'].str.split(',', n=1, expand=True)
actual_cols_num = temp.shape[1]
prefix = '商品名称'
dynamic_names_1 = [f"{prefix}{i+1}" for i in range(actual_cols_num)]
split_result_renamed1 = temp.rename(columns=dict(zip(temp.columns, dynamic_names_1)))
m_data = pd.concat([m_data.drop('商品名称', axis=1), split_result_renamed1], axis=1)
m_data = m_data[['订单编号', '商品名称1', '销量']]
m_data = m_data.rename(columns={'商品名称1':'商品名称'})
m_pro_data = pd.merge(m_data, p_data, how='left', left_on='商品名称', right_on='名称')
m_pro_data = m_pro_data.drop(columns=['名称'])
m_pro_data = m_pro_data.dropna(subset=['商品名称'])
m_pro_data['销量'] = m_pro_data['销量'].astype(int)
m_pro_data['线下营业额'] = m_pro_data['线下售价'] * m_pro_data['销量']
m_pro_data['成本金额'] = m_pro_data['成本'] * m_pro_data['销量']
m_pro_data = m_pro_data.drop(columns=['线下售价', '成本', '线上售价'])
m_sum_data = m_pro_data.groupby('订单编号')[['线下营业额', '成本金额']].sum()
m_sum_data = m_sum_data.reset_index()

m_profit = pd.merge(account_data, m_sum_data, how='inner', right_on='订单编号', left_on='订单单号')
m_profit = m_profit.drop(columns='订单编号')
m_profit['结算占比'] = m_profit['结算金额'] / m_profit['线下营业额']
m_profit['利润'] = m_profit['结算金额'] - m_profit['成本金额']
m_profit['利润率'] = np.where(m_profit['营业额'] == 0, 0, m_profit['利润'] / m_profit['营业额'])
m_profit['订单单号'] = m_profit['订单单号'].astype(str)
m_profit = m_profit.rename(columns={'门店id': '门店ID'})
kp_mt_profit = pd.merge(m_profit, kp_info[['门店名称', '美团门店ID', '城市']], how='left', left_on='门店ID', right_on='美团门店ID')
kp_mt_profit['平台'] = '美团'
kp_mt_profit = kp_mt_profit.drop(columns=['门店名称_x', '美团门店ID'])
kp_mt_profit = kp_mt_profit.rename(columns={'门店名称_y': '门店名称'})

# 饿了么数据处理
fpath2 = '../data/品牌1饿了么财务数据.xlsx'
a_data = pd.read_excel(fpath2, sheet_name='外卖账单明细', parse_dates=['账单日期'])
time_column = a_data['账单日期'].values
a_data['账单日期'] = np.array(time_column, dtype='datetime64')
a_data = a_data[['门店ID', '门店名称', '账单日期', '订单编号', '结算金额', '商品金额', '打包费']]
a_data['营业额'] = a_data['商品金额'] + a_data['打包费']
a_sum_data = a_data.groupby('订单编号')[['结算金额', '营业额']].sum()
a_sum_data = a_sum_data.reset_index()
a_sum_data = a_sum_data.rename(columns={'订单编号': '订单单号'})
account_data = pd.merge(a_sum_data, a_data[['门店ID', '门店名称', '账单日期', '订单编号']], how='left', left_on='订单单号', right_on='订单编号')
account_data = account_data.drop_duplicates(subset=['订单单号'],keep='first')
account_data = account_data.drop(columns='订单编号')

fpath3 = '../data/品牌1饿了么订单数据.xlsx'
e_data = pd.read_excel(fpath3, sheet_name='data')
e_data = e_data[['订单单号', '商品信息']]
temp = e_data['商品信息'].str.split(r'\d\+',expand=True)
actual_cols_num = temp.shape[1]
prefix = '商品名称'
dynamic_names_1 = [f"{prefix}{i+1}" for i in range(actual_cols_num)]
split_result_renamed1 = temp.rename(columns=dict(zip(temp.columns, dynamic_names_1)))
e_data = pd.concat([e_data.drop('商品信息', axis=1), split_result_renamed1], axis=1)
e_data = e_data.melt(
    id_vars=['订单单号'],
    var_name='商品名称',
    value_name='商品信息'
)
e_data = e_data.drop(columns=['商品名称'])
temp = e_data['商品信息'].str.split('*',expand=True)
actual_cols_num = temp.shape[1]
prefix = '商品名称'
dynamic_names_1 = [f"{prefix}{i+1}" for i in range(actual_cols_num)]
split_result_renamed1 = temp.rename(columns=dict(zip(temp.columns, dynamic_names_1)))
e_data = pd.concat([e_data.drop('商品信息', axis=1), split_result_renamed1], axis=1)
e_data = e_data[['订单单号', '商品名称1']]
e_data = e_data.rename(columns={'商品名称1':'商品信息'})
temp = e_data['商品信息'].str.split('_',expand=True)
actual_cols_num = temp.shape[1]
prefix = '商品名称'
dynamic_names_1 = [f"{prefix}{i+1}" for i in range(actual_cols_num)]
split_result_renamed1 = temp.rename(columns=dict(zip(temp.columns, dynamic_names_1)))
# pandas3: str 列不能 iloc 原地写入 float，整列替换
for _col in list(split_result_renamed1.columns[1:]):
    split_result_renamed1[_col] = pd.to_numeric(split_result_renamed1[_col], errors='coerce')
e_data = pd.concat([e_data.drop('商品信息', axis=1), split_result_renamed1], axis=1)
e_data['销量'] = e_data.iloc[:, 2:].sum(axis=1)
e_data = e_data.rename(columns={'商品名称1':'商品名称'})
e_data = e_data[['订单单号', '商品名称', '销量']]
temp = e_data['商品名称'].str.split('[',expand=True)
actual_cols_num = temp.shape[1]
prefix = '商品名称'
dynamic_names_1 = [f"{prefix}{i+1}" for i in range(actual_cols_num)]
split_result_renamed1 = temp.rename(columns=dict(zip(temp.columns, dynamic_names_1)))
e_data = pd.concat([e_data.drop('商品名称', axis=1), split_result_renamed1], axis=1)
e_data = e_data[['订单单号', '商品名称1', '销量']]
e_data = e_data.rename(columns={'商品名称1':'商品名称'})
e_pro_data = pd.merge(e_data, p_data, how='left', left_on='商品名称', right_on='名称')
e_pro_data = e_pro_data.drop(columns=['名称'])
e_pro_data = e_pro_data.dropna(subset=['商品名称'])
e_pro_data['销量'] = e_pro_data['销量'].astype(int)
e_pro_data['线下营业额'] = e_pro_data['线下售价'] * e_pro_data['销量']
e_pro_data['成本金额'] = e_pro_data['成本'] * e_pro_data['销量']
e_pro_data = e_pro_data.rename(columns={'订单单号': '订单编号'})
e_pro_data = e_pro_data.drop(columns=['线下售价', '成本', '线上售价'])
e_sum_data = e_pro_data.groupby('订单编号')[['线下营业额', '成本金额']].sum()
e_sum_data = e_sum_data.reset_index()
#
e_profit = pd.merge(account_data, e_sum_data, how='inner', right_on='订单编号', left_on='订单单号')
e_profit = e_profit.drop(columns='订单编号')
e_profit['结算占比'] = e_profit['结算金额'] / e_profit['线下营业额']
e_profit['利润'] = e_profit['结算金额'] - e_profit['成本金额']
e_profit['利润率'] = np.where(e_profit['营业额'] == 0, 0, e_profit['利润'] / e_profit['营业额'])
e_profit['订单单号'] = e_profit['订单单号'].astype(str)
kp_elm_profit = pd.merge(e_profit, kp_info[['门店名称', '饿了么门店ID', '城市']], how='left', left_on='门店ID', right_on='饿了么门店ID')
kp_elm_profit['平台'] = '饿了么'
kp_elm_profit = kp_elm_profit.drop(columns=['门店名称_x', '饿了么门店ID'])
kp_elm_profit = kp_elm_profit.rename(columns={'门店名称_y': '门店名称'})

# 合并数据
kp_profit = pd.concat([kp_mt_profit, kp_elm_profit], ignore_index=True)


In [ ]:
# 品牌1订单数据处理
# 美团数据
fpath1 = "../data/品牌1美团订单数据.csv"
m_data = pd.read_csv(fpath1, encoding = 'gbk', parse_dates=['下单时间'])

time_column = m_data['下单时间'].values
m_data['下单时间'] = np.array(time_column, dtype='datetime64')
# 先判活动再 fillna，避免空活动被填成 0 后全变成「是」
m_data['活动信息'] = m_data['活动信息'].fillna('').astype(str).str.strip()
m_data['是否参加活动'] = np.where(m_data['活动信息'] == '', '否', '是')
num_cols = m_data.select_dtypes(include=[np.number]).columns
m_data[num_cols] = m_data[num_cols].fillna(0)
m_data['订单金额'] = m_data['商品原价'] + m_data['包装费']
m_data = m_data[m_data['订单状态']!='已取消']
m_data = m_data[['订单编号','门店id','下单时间','订单金额', '订单实付', '是否参加活动']]
m_data['平台'] = '美团'
# 餐饮时间段划分：早餐（23:00-10:00），午餐（10:00-14:00），晚餐（14:00-23:00）
m_data['下单分类'] = np.where((m_data['下单时间'].dt.hour >= 23) | (m_data['下单时间'].dt.hour < 10), '早餐',np.where((m_data['下单时间'].dt.hour >= 10) & (m_data['下单时间'].dt.hour < 14), '午餐',np.where((m_data['下单时间'].dt.hour >= 14) & (m_data['下单时间'].dt.hour < 23), '晚餐', '其他')))
# 按订单金额进行分类
m_data['下单金额分类'] = np.where((m_data['订单金额'] >= 0) & (m_data['订单金额'] < 40), '0-40元',np.where((m_data['订单金额'] >= 40) & (m_data['订单金额'] < 60), '40-60元',np.where((m_data['订单金额'] >= 60) & (m_data['订单金额'] < 85), '60-85元',np.where((m_data['订单金额'] >= 85), '85元以上', '其他'))))
# 按订单实付进行分类
m_data['订单实付分类'] = np.where((m_data['订单实付'] >= 0) & (m_data['订单实付'] < 40), '0-40元',np.where((m_data['订单实付'] >= 40) & (m_data['订单实付'] < 60), '40-60元',np.where((m_data['订单实付'] >= 60) & (m_data['订单实付'] < 85), '60-85元',np.where((m_data['订单实付'] >= 85), '85元以上', '其他'))))

# 连接数据
kp_mt_order = pd.merge(m_data, kp_info[['门店名称', '美团门店ID', '城市']], how='left', left_on='门店id', right_on='美团门店ID')
kp_mt_order = kp_mt_order.drop(columns=['美团门店ID','订单编号'])


# 饿了么数据
fpath3 = '../data/品牌1饿了么订单数据.xlsx'
e_data = pd.read_excel(fpath3, sheet_name='data', parse_dates=['下单时间'])
time_column = e_data['下单时间'].values
e_data['下单时间'] = np.array(time_column, dtype='datetime64')
e_data['活动信息'] = e_data['活动信息'].fillna('').astype(str).str.strip()
e_data['是否参加活动'] = np.where(e_data['活动信息'] == '', '否', '是')
num_cols = e_data.select_dtypes(include=[np.number]).columns
e_data[num_cols] = e_data[num_cols].fillna(0)
e_data['订单金额'] = e_data['菜品原价'] + e_data['餐盒费']
e_data = e_data[e_data['订单状态']!='订单无效']
e_data = e_data[['订单单号','门店编号','下单时间','订单金额', '顾客实付', '是否参加活动']]
e_data['平台'] = '饿了么'
# 餐饮时间段划分：早餐（23:00-10:00），午餐（10:00-14:00），晚餐（14:00-23:00）
e_data['下单分类'] = np.where((e_data['下单时间'].dt.hour >= 23) | (e_data['下单时间'].dt.hour < 10), '早餐',np.where((e_data['下单时间'].dt.hour >= 10) & (e_data['下单时间'].dt.hour < 14), '午餐',np.where((e_data['下单时间'].dt.hour >= 14) & (e_data['下单时间'].dt.hour < 23), '晚餐', '其他')))
# 按订单金额进行分类
e_data['下单金额分类'] = np.where((e_data['订单金额'] >= 0) & (e_data['订单金额'] < 40), '0-40元',np.where((e_data['订单金额'] >= 40) & (e_data['订单金额'] < 60), '40-60元',np.where((e_data['订单金额'] >= 60) & (e_data['订单金额'] < 85), '60-85元',np.where((e_data['订单金额'] >= 85), '85元以上', '其他'))))
# 按订单实付进行分类
e_data['订单实付分类'] = np.where((e_data['顾客实付'] >= 0) & (e_data['顾客实付'] < 40), '0-40元',np.where((e_data['顾客实付'] >= 40) & (e_data['顾客实付'] < 60), '40-60元',np.where((e_data['顾客实付'] >= 60) & (e_data['顾客实付'] < 85), '60-85元',np.where((e_data['顾客实付'] >= 85), '85元以上', '其他'))))

# 连接数据
kp_elm_order = pd.merge(e_data, kp_info[['门店名称', '饿了么门店ID', '城市']], how='left', left_on='门店编号', right_on='饿了么门店ID')
# kp_elm_order['订单单号'] = kp_elm_order['订单单号'].astype(str)
# kp_elm_profit = kp_elm_profit.rename(columns={"订单单号": '订单编号'})
# kp_elm_order = pd.merge(kp_elm_order, kp_elm_profit[['订单编号', '结算占比', '利润率']], how="left", left_on='订单单号', right_on='订单编号')
kp_elm_order = kp_elm_order.rename(columns={'门店编号':'门店id', '顾客实付':'订单实付'})
kp_elm_order = kp_elm_order.drop(columns=['饿了么门店ID', '订单单号'])
# kp_elm_order['利润率分类'] = np.where(
#     (kp_elm_order['利润率'] < 0) , '负利润订单',
#     np.where((kp_elm_order['利润率'] >= 0) & (kp_elm_order['利润率'] < 0.15), '0-0.15',
#              np.where((kp_elm_order['利润率'] >= 0.15) & (kp_elm_order['利润率'] < 0.25), '0.15-0.25',
#                       np.where((kp_elm_order['利润率'] >= 0.25) & (kp_elm_order['利润率'] < 0.35), '0.25-0.35',
#                                np.where((kp_elm_order['利润率'] >= 0.35), '0.35以上', '其他')))))

kp_order = pd.concat([kp_mt_order, kp_elm_order], ignore_index=True)
kp_order['下单时间'] = kp_order['下单时间'].dt.normalize()


In [ ]:
# 品牌1营销活动拆解
# 美团营业额 = 商品原价 + 包装费；饿了么营业额 = 菜品原价 + 餐盒费
# 一笔订单参加多个活动时，营业额和订单数分别记入每个活动；商家活动支出只记该活动的商家承担
import re

_MT_MERCHANT = re.compile(r"商家承担[:：]\s*(\d+(?:\.\d+)?)")
_MT_GOODS = re.compile(r"购买(.+?)原价\d+(?:\.\d+)?元现价(\d+(?:\.\d+)?)元")
_MT_QTY = re.compile(r",单价(\d+(?:\.\d+)?)\*数量(\d+(?:\.\d+)?)")
_ELM_PART = re.compile(r"【(.+?)】:([^【]*)")
_ELM_MERCHANT = re.compile(r"商户补贴\s*(\d+(?:\.\d+)?)元")

def _blank(raw):
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return ""
    text = str(raw).strip()
    return "" if text.lower() == "nan" else text

def _mt_category(part):
    text = part.strip()
    if text.startswith("减配送费"):
        return "减配送费"
    if text.startswith("减配追补"):
        return "减配追补"
    if text.startswith("用户使用了支付红包"):
        return "支付红包"
    if text.startswith("用户使用了商家代金券"):
        return "商家代金券"
    if text.startswith("商品折扣券"):
        return "商品折扣券"
    if text.startswith("购买"):
        return "商品折扣"
    if text.startswith("满") and "减" in text:
        return "满减"
    return "其他"

def _mt_discount_qty(part, goods):
    # 文案里的商家承担是单件金额，按商品信息里同名同现价的数量放大
    found = _MT_GOODS.search(part)
    if not found or not goods:
        return 1.0
    name, price = found.group(1), found.group(2)
    qty = 0.0
    for item in goods.split("/"):
        item = item.strip()
        if not item.startswith(name):
            continue
        hit = _MT_QTY.search(item)
        if hit and hit.group(1) == price:
            qty += float(hit.group(2))
    return qty if qty > 0 else 1.0

def _parse_meituan(raw, goods=""):
    text = _blank(raw)
    if not text:
        return []
    goods_text = _blank(goods)
    merged = {}
    for part in text.split("/"):
        part = part.strip()
        if not part:
            continue
        cat = _mt_category(part)
        spend = sum(float(x) for x in _MT_MERCHANT.findall(part))
        if cat in ("商品折扣", "商品折扣券"):
            spend *= _mt_discount_qty(part, goods_text)
        merged[cat] = merged.get(cat, 0.0) + spend
    return list(merged.items())

def _parse_eleme(raw, goods=""):
    text = _blank(raw)
    if not text:
        return []
    merged = {}
    for cat, body in _ELM_PART.findall(text):
        # 饿了么「店铺满减」与美团「满X减Y」是同一类活动
        if cat == "店铺满减":
            cat = "满减"
        spend = sum(float(x) for x in _ELM_MERCHANT.findall(body))
        merged[cat] = merged.get(cat, 0.0) + spend
    return list(merged.items())

def _explode_activity(orders, parser):
    rows = []
    for rec in orders.itertuples(index=False):
        for cat, spend in parser(rec.活动信息, rec.商品信息):
            rows.append({
                "日期": rec.日期,
                "平台": rec.平台,
                "门店id": rec.门店id,
                "活动分类": cat,
                "营业额": rec.营业额,
                "商家活动支出": spend,
                "订单编号": rec.订单编号,
            })
    detail = pd.DataFrame(rows)
    grouped = (
        detail.groupby(["日期", "平台", "门店id", "活动分类"], as_index=False)
        .agg(营业额=("营业额", "sum"), 商家活动支出=("商家活动支出", "sum"), 订单数=("订单编号", "nunique"))
    )
    grouped["营业额"] = grouped["营业额"].round(2)
    grouped["商家活动支出"] = grouped["商家活动支出"].round(2)
    grouped["订单数"] = grouped["订单数"].astype(int)
    return grouped

# 美团订单
fpath1 = "../data/品牌1美团订单数据.csv"
m_act = pd.read_csv(fpath1, encoding="gbk", parse_dates=["下单时间"])
m_act = m_act[m_act["订单状态"] != "已取消"].copy()
m_act["日期"] = m_act["下单时间"].dt.normalize()
m_act["营业额"] = m_act["商品原价"].fillna(0) + m_act["包装费"].fillna(0)
m_act["平台"] = "美团"
m_act["活动信息"] = m_act["活动信息"].fillna("").astype(str).str.strip()
m_act["商品信息"] = m_act["商品信息"].fillna("").astype(str)
kp_mt_activity = _explode_activity(m_act, _parse_meituan)
kp_mt_activity = pd.merge(
    kp_mt_activity,
    kp_info[["门店名称", "美团门店ID", "城市"]].drop_duplicates("美团门店ID"),
    how="left", left_on="门店id", right_on="美团门店ID",
)
kp_mt_activity = kp_mt_activity.drop(columns=["美团门店ID"])

# 饿了么订单
fpath3 = "../data/品牌1饿了么订单数据.xlsx"
e_act = pd.read_excel(fpath3, sheet_name="data", parse_dates=["下单时间"])
e_act = e_act[e_act["订单状态"] != "订单无效"].copy()
e_act["日期"] = e_act["下单时间"].dt.normalize()
e_act["营业额"] = e_act["菜品原价"].fillna(0) + e_act["餐盒费"].fillna(0)
e_act["平台"] = "饿了么"
e_act["门店id"] = e_act["门店编号"]
e_act["订单编号"] = e_act["订单单号"]
e_act["活动信息"] = e_act["活动信息"].fillna("").astype(str).str.strip()
e_act["商品信息"] = e_act["商品信息"].fillna("").astype(str)
kp_elm_activity = _explode_activity(e_act, _parse_eleme)
kp_elm_activity = pd.merge(
    kp_elm_activity, kp_info[["门店名称", "饿了么门店ID", "城市"]],
    how="left", left_on="门店id", right_on="饿了么门店ID",
)
kp_elm_activity = kp_elm_activity.drop(columns=["饿了么门店ID"])

kp_activity = pd.concat([kp_mt_activity, kp_elm_activity], ignore_index=True)
kp_activity = kp_activity[["日期", "平台", "门店id", "门店名称", "城市", "活动分类", "营业额", "商家活动支出", "订单数"]]
kp_activity = kp_activity.sort_values(["日期", "平台", "门店名称", "活动分类"]).reset_index(drop=True)


In [ ]:
# 品牌1流量数据
# 美团流量数据表
fpath = "../data/品牌1美团数据.csv"
m_data = pd.read_csv(fpath, encoding = 'gbk', parse_dates=['日期'])
time_column = m_data['日期'].values
m_data['日期'] = np.array(time_column, dtype='datetime64')
m_data = m_data.fillna(0)
m_data = m_data[['日期','门店id','曝光人数','入店人数','下单人数', '下单新客', '下单老客']]
kp_mt_exposure = pd.merge(m_data, kp_info[['门店名称', '美团门店ID', '城市']], how='left', left_on='门店id', right_on='美团门店ID')
kp_mt_exposure['平台'] = '美团'
kp_mt_exposure = kp_mt_exposure.drop(columns=['美团门店ID'])

# 饿了么数据
fpath1 = "../data/品牌1饿了么数据.xlsx"
e_data = pd.read_excel(fpath1, sheet_name='data', parse_dates=['日期'])
time_column = e_data['日期'].values
e_data['日期'] = np.array(time_column, dtype='datetime64')
e_data = e_data.fillna(0)
e_data = e_data[['日期','门店编号','曝光人数','进店人数','下单人数', '新客下单人数', '老客下单人数']]
kp_elm_exposure = pd.merge(e_data, kp_info[['门店名称', '饿了么门店ID', '城市']], how='left', left_on='门店编号', right_on='饿了么门店ID')
kp_elm_exposure['平台'] = '饿了么'
kp_elm_exposure = kp_elm_exposure.drop(columns=['饿了么门店ID'])
kp_elm_exposure = kp_elm_exposure.rename(columns={'门店编号':'门店id', '进店人数':'入店人数', '新客下单人数':'下单新客', '老客下单人数':'下单老客'})

# 合并数据；列名与 backend config.COLUMNS['exposure'] 对齐
kp_exposure = pd.concat([kp_mt_exposure, kp_elm_exposure], ignore_index=True)
kp_exposure = kp_exposure.rename(columns={'下单新客': '下单新客人数', '下单老客': '下单老客人数'})


In [ ]:
# 品牌1商品数据处理
# 美团商品数据处理
fpath = "../data/品牌1美团商品数据.csv"
m_data = pd.read_csv(fpath, encoding = 'gbk', parse_dates=['日期'])
time_column = m_data['日期'].values
m_data['日期'] = np.array(time_column, dtype='datetime64')
m_data = m_data[['日期', '门店id','商品名','商品销量','商品销售额']]
kp_mt_product = pd.merge(m_data, kp_info[['门店名称', '美团门店ID', '城市']], how='left', left_on='门店id', right_on='美团门店ID')
kp_mt_product = kp_mt_product.drop(columns=['美团门店ID'])
kp_mt_product['平台'] = '美团'
kp_mt_product = kp_mt_product.rename(columns={'商品名':'商品名称'})

# 饿了么商品数据处理
fpath1 = "../data/品牌1饿了么商品数据.xlsx"
e_data = pd.read_excel(fpath1, sheet_name='data')
e_data['日期'] = e_data['日期'].astype(str).str[:10]
e_data['日期'] = pd.to_datetime(e_data['日期'], errors='coerce').dt.normalize()
e_data = e_data[e_data['销量'] > 0] # 过滤销量大于0的数据
e_data = e_data[['日期', '门店编号','商品名称','销量','销售额']]
kp_elm_product = pd.merge(e_data, kp_info[['门店名称', '饿了么门店ID', '城市']], how='left', left_on='门店编号', right_on='饿了么门店ID')
kp_elm_product = kp_elm_product.drop(columns=['饿了么门店ID'])
kp_elm_product = kp_elm_product.rename(columns={'门店编号':'门店id'})
kp_elm_product['平台'] = '饿了么'
kp_elm_product = kp_elm_product.rename(columns={'销量':'商品销量', '销售额':'商品销售额'})

# 合并数据
kp_product = pd.concat([kp_mt_product, kp_elm_product], ignore_index=True)


In [ ]:
# 品牌3 财务数据处理
# 美团数据表
fpath1 = "../data/品牌3美团数据.csv"
m_data = pd.read_csv(fpath1, encoding = 'gbk', parse_dates=['日期'])
time_column = m_data['日期'].values
m_data['日期'] = np.array(time_column, dtype='datetime64')
m_data[m_data.select_dtypes(include=['number']).columns] = m_data.select_dtypes(include=['number']).fillna(0)
m_data['营业额'] = m_data['商品原价'] + m_data['包装费']
m_data['平台佣金'] = m_data['平台服务费(含佣金和配送服务费)']
use_m_data1 = m_data[['日期','门店id','营业额','有效订单','平台佣金','商家活动支出','营业收入']]
zz_mt_account = use_m_data1.rename(columns={'有效订单':'有效订单数', '商家活动支出':'商家补贴金额', '营业收入':'收入'})
zz_mt_account['平台'] = '美团'
# 连接门店信息表
zz_mt_account = pd.merge(zz_mt_account, zz_info[['门店名称', '美团门店ID', '大区']], how='left', left_on='门店id', right_on='美团门店ID')
zz_mt_account = zz_mt_account.drop(columns=['美团门店ID'])

# 饿了么数据表
fpath2 = "../data/品牌3饿了么数据.xlsx"
e_data = pd.read_excel(fpath2, sheet_name='data', parse_dates=['日期'])
time_column = e_data['日期'].values
e_data['日期'] = pd.to_datetime(e_data['日期'], errors='coerce').dt.normalize()
e_data[e_data.select_dtypes(include=['number']).columns] = e_data.select_dtypes(include=['number']).fillna(0)
e_data['营业额'] = e_data['商品销售额'] + e_data['打包费']
e_data['平台佣金'] = e_data['平台技术服务费'] + e_data['履约技术服务费']
use_e_data1 = e_data[['日期','门店编号','营业额','有效订单','平台佣金','商家活动成本（含满减活动）','收入']]
zz_elm_account = use_e_data1.rename(columns={'门店编号':'门店id','有效订单':'有效订单数', '商家活动成本（含满减活动）':'商家补贴金额'})
zz_elm_account['平台'] = '饿了么'
zz_elm_account = pd.merge(zz_elm_account, zz_info[['门店名称', '饿了么门店ID', '大区']], how='left', left_on='门店id', right_on='饿了么门店ID')
zz_elm_account = zz_elm_account.drop(columns=['饿了么门店ID'])

# 合并数据
zz_account = pd.concat([zz_mt_account, zz_elm_account], ignore_index=True)


In [ ]:
# # 品牌3 订单数据处理
# # 美团数据
fpath1 = "../data/品牌3美团订单数据.csv"
m_data = pd.read_csv(fpath1, encoding = 'gbk', parse_dates=['下单时间'])

time_column = m_data['下单时间'].values
m_data['下单时间'] = np.array(time_column, dtype='datetime64')
m_data['活动信息'] = m_data['活动信息'].fillna('').astype(str).str.strip()
m_data['是否参加活动'] = np.where(m_data['活动信息'] == '', '否', '是')
num_cols = m_data.select_dtypes(include=[np.number]).columns
m_data[num_cols] = m_data[num_cols].fillna(0)
m_data['订单金额'] = m_data['商品原价'] + m_data['包装费']
m_data = m_data[m_data['订单状态']!='已取消']
m_data = m_data[['订单编号','门店id','下单时间','订单金额', '订单实付', '是否参加活动']]
m_data['平台'] = '美团'
# 餐饮时间段划分：早餐（23:00-10:00），午餐（10:00-14:00），晚餐（14:00-23:00）
m_data['下单分类'] = np.where((m_data['下单时间'].dt.hour >= 23) | (m_data['下单时间'].dt.hour < 10), '早餐',np.where((m_data['下单时间'].dt.hour >= 10) & (m_data['下单时间'].dt.hour < 14), '午餐',np.where((m_data['下单时间'].dt.hour >= 14) & (m_data['下单时间'].dt.hour < 23), '晚餐', '其他')))
# 按订单金额进行分类
m_data['下单金额分类'] = np.where((m_data['订单金额'] >= 0) & (m_data['订单金额'] < 15), '0-15元',np.where((m_data['订单金额'] >= 15) & (m_data['订单金额'] < 20), '15-20元',np.where((m_data['订单金额'] >= 20) & (m_data['订单金额'] < 25), '20-25元',np.where((m_data['订单金额'] >= 25), '25元以上', '其他'))))
# 按订单实付进行分类
m_data['订单实付分类'] = np.where((m_data['订单实付'] >= 0) & (m_data['订单实付'] < 15), '0-15元',np.where((m_data['订单实付'] >= 15) & (m_data['订单实付'] < 20), '15-20元',np.where((m_data['订单实付'] >= 20) & (m_data['订单实付'] < 25), '20-25元',np.where((m_data['订单实付'] >= 25), '25元以上', '其他'))))

# 连接数据
zz_mt_order = pd.merge(m_data, zz_info[['门店名称', '美团门店ID', '大区']], how='left', left_on='门店id', right_on='美团门店ID')
zz_mt_order = zz_mt_order.drop(columns=['美团门店ID'])


# 饿了么数据
fpath3 = '../data/品牌3饿了么订单数据.xlsx'
e_data = pd.read_excel(fpath3, sheet_name='data', parse_dates=['下单时间'])
time_column = e_data['下单时间'].values
e_data['下单时间'] = np.array(time_column, dtype='datetime64')
e_data['活动信息'] = e_data['活动信息'].fillna('').astype(str).str.strip()
e_data['是否参加活动'] = np.where(e_data['活动信息'] == '', '否', '是')
num_cols = e_data.select_dtypes(include=[np.number]).columns
e_data[num_cols] = e_data[num_cols].fillna(0)
e_data['订单金额'] = e_data['菜品原价'] + e_data['餐盒费']
e_data = e_data[e_data['订单状态']!='订单无效']
e_data = e_data[['订单单号','门店编号','下单时间','订单金额', '顾客实付', '是否参加活动']]
e_data['平台'] = '饿了么'
# 餐饮时间段划分：早餐（23:00-10:00），午餐（10:00-14:00），晚餐（14:00-23:00）
e_data['下单分类'] = np.where((e_data['下单时间'].dt.hour >= 23) | (e_data['下单时间'].dt.hour < 10), '早餐',np.where((e_data['下单时间'].dt.hour >= 10) & (e_data['下单时间'].dt.hour < 14), '午餐',np.where((e_data['下单时间'].dt.hour >= 14) & (e_data['下单时间'].dt.hour < 23), '晚餐', '其他')))
# 按订单金额进行分类
e_data['下单金额分类'] = np.where((e_data['订单金额'] >= 0) & (e_data['订单金额'] < 15), '0-15元',np.where((e_data['订单金额'] >= 15) & (e_data['订单金额'] < 20), '15-20元',np.where((e_data['订单金额'] >= 20) & (e_data['订单金额'] < 25), '20-25元',np.where((e_data['订单金额'] >= 25), '25元以上', '其他'))))
# 按订单实付进行分类
e_data['订单实付分类'] = np.where((e_data['顾客实付'] >= 0) & (e_data['顾客实付'] < 15), '0-15元',np.where((e_data['顾客实付'] >= 15) & (e_data['顾客实付'] < 20), '15-20元',np.where((e_data['顾客实付'] >= 20) & (e_data['顾客实付'] < 25), '20-25元',np.where((e_data['顾客实付'] >= 25), '25元以上', '其他'))))

# 连接数据
zz_elm_order = pd.merge(e_data, zz_info[['门店名称', '饿了么门店ID', '大区']], how='left', left_on='门店编号', right_on='饿了么门店ID')
zz_elm_order = zz_elm_order.drop(columns=['饿了么门店ID'])
zz_elm_order = zz_elm_order.rename(columns={'订单单号':'订单编号','门店编号':'门店id', '顾客实付':'订单实付'})

zz_order = pd.concat([zz_mt_order,zz_elm_order], ignore_index=True)


In [ ]:
# 品牌3 流量数据
# 美团流量数据表
fpath = "../data/品牌3美团数据.csv"
m_data = pd.read_csv(fpath, encoding = 'gbk', parse_dates=['日期'])
time_column = m_data['日期'].values
m_data['日期'] = np.array(time_column, dtype='datetime64')
m_data = m_data.fillna(0)
m_data = m_data[['日期','门店id','曝光人数','入店人数','下单人数', '下单新客', '下单老客']]
zz_mt_exposure = pd.merge(m_data, zz_info[['门店名称', '美团门店ID', '大区']], how='left', left_on='门店id', right_on='美团门店ID')
zz_mt_exposure['平台'] = '美团'
zz_mt_exposure = zz_mt_exposure.drop(columns=['美团门店ID'])

# 饿了么数据
fpath1 = "../data/品牌3饿了么数据.xlsx"
e_data = pd.read_excel(fpath1, sheet_name='data', parse_dates=['日期'])
time_column = e_data['日期'].values
e_data['日期'] = pd.to_datetime(e_data['日期'], errors='coerce').dt.normalize()
e_data = e_data.fillna(0)
e_data = e_data[['日期','门店编号','曝光人数','进店人数','下单人数', '新客下单人数', '老客下单人数']]
zz_elm_exposure = pd.merge(e_data, zz_info[['门店名称', '饿了么门店ID', '大区']], how='left', left_on='门店编号', right_on='饿了么门店ID')
zz_elm_exposure['平台'] = '饿了么'
zz_elm_exposure = zz_elm_exposure.drop(columns=['饿了么门店ID'])
zz_elm_exposure = zz_elm_exposure.rename(columns={'门店编号':'门店id', '进店人数':'入店人数', '新客下单人数':'下单新客', '老客下单人数':'下单老客'})

# 合并数据；列名与 backend config 对齐
zz_exposure = pd.concat([zz_mt_exposure, zz_elm_exposure], ignore_index=True)
zz_exposure = zz_exposure.rename(columns={'下单新客': '下单新客人数', '下单老客': '下单老客人数'})


In [ ]:
# 品牌3 商品数据处理
# 美团商品数据处理
fpath = "../data/品牌3美团商品数据.csv"
m_data = pd.read_csv(fpath, encoding = 'gbk', parse_dates=['日期'])
time_column = m_data['日期'].values
m_data['日期'] = np.array(time_column, dtype='datetime64')
m_data = m_data[['日期', '门店id','商品名','商品销量','商品销售额']]
zz_mt_product = pd.merge(m_data, zz_info[['门店名称', '美团门店ID', '大区']], how='left', left_on='门店id', right_on='美团门店ID')
zz_mt_product = zz_mt_product.drop(columns=['美团门店ID'])
zz_mt_product['平台'] = '美团'
zz_mt_product = zz_mt_product.rename(columns={'商品名':'商品名称'})
# 饿了么商品数据处理
fpath1 = "../data/品牌3饿了么商品数据.xlsx"
e_data = pd.read_excel(fpath1, sheet_name='data')
e_data['日期'] = e_data['日期'].astype(str).str[:10]
e_data['日期'] = pd.to_datetime(e_data['日期'], errors='coerce').dt.normalize()
e_data = e_data[e_data['销量'] > 0] # 过滤销量大于0的数据
e_data = e_data[['日期', '门店编号','商品名称','销量','销售额']]
zz_elm_product = pd.merge(e_data, zz_info[['门店名称', '饿了么门店ID', '大区']], how='left', left_on='门店编号', right_on='饿了么门店ID')
zz_elm_product = zz_elm_product.drop(columns=['饿了么门店ID'])
zz_elm_product = zz_elm_product.rename(columns={'门店编号':'门店id'})
zz_elm_product['平台'] = '饿了么'
zz_elm_product = zz_elm_product.rename(columns={'销量':'商品销量', '销售额':'商品销售额'})

# 合并数据
zz_product = pd.concat([zz_mt_product, zz_elm_product], ignore_index=True)

In [ ]:
# 导入机构信息表
# fpath = "../data/餐饮门店表.xlsx"
# cy_info = pd.read_excel(fpath, sheet_name='机构汇总')

path1 = "../data/品牌2门店表.xlsx"
xj_info = pd.read_excel(path1, sheet_name='机构汇总')

path2 = "../data/品牌3门店表.xlsx"
zz_info = pd.read_excel(path2, sheet_name='机构汇总')

fpath3 = "../data/品牌1门店表.xlsx"
kp_info = pd.read_excel(fpath3, sheet_name='机构汇总')


In [ ]:
if "IMPORT_BATCH" not in globals():
    IMPORT_BATCH = begin_import_batch()

import re

def _abnormal_issues(raw):
    text = "" if raw is None or (isinstance(raw, float) and pd.isna(raw)) else str(raw)
    parts = [p.replace("\t", "").strip() for p in re.split(r"[；;]+", text)]
    return [p for p in parts if p and p != "--" and p.lower() != "nan"]

_abn = None
for _enc in ("utf-8-sig", "gbk"):
    try:
        _abn = pd.read_csv("../data/美团异常订单数据.csv", encoding=_enc, dtype=str)
        break
    except UnicodeDecodeError:
        continue
if _abn is None:
    raise RuntimeError("无法读取美团异常订单数据.csv")

def _abn_cell(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    return str(value).replace("\t", "").strip()

_abn_rows = []
for _rec in _abn.to_dict("records"):
    _when = _abn_cell(_rec.get("下单时间"))
    if not _when:
        continue
    for _issue in _abnormal_issues(_rec.get("商家服务问题")):
        _abn_rows.append({
            "订单编号": _abn_cell(_rec.get("订单编号")),
            "下单时间": pd.to_datetime(_when),
            "店铺ID": _abn_cell(_rec.get("店铺ID")),
            "门店名称": _abn_cell(_rec.get("店铺名称")),
            "平台": "美团",
            "商家服务问题": _issue,
        })
xj_abnormal = pd.DataFrame(_abn_rows)

# 品牌2 表格导入 MySQL（按日覆盖 append；info 整表 replace）
# 密码：export MYSQL_PASSWORD='***'
# 若旧表仍是 TEXT/FLOAT、无业务索引，首次请加 force_rebuild=True 重建表结构
with make_exporter() as exporter:
    results = export_brand_tables(exporter, {"xj_account": xj_account, "xj_info": xj_info, "xj_order": xj_order, "xj_exposure": xj_exposure, "xj_product": xj_product, "xj_abnormal": xj_abnormal}, chunksize=5000, force_rebuild=False, batch_id=IMPORT_BATCH, finalize_batch=False)
    for name, ok in results.items():
        print(f"{'OK' if ok else 'FAIL'}: {name}")


In [ ]:
# 品牌1 表格导入 MySQL（按日覆盖 append；info 整表 replace）
# 密码：export MYSQL_PASSWORD='***'
# 若旧表仍是 TEXT/FLOAT、无业务索引，首次请加 force_rebuild=True 重建表结构
with make_exporter() as exporter:
    results = export_brand_tables(exporter, {"kp_account": kp_account, "kp_info": kp_info, "kp_order": kp_order, "kp_exposure": kp_exposure, "kp_product": kp_product, "kp_profit": kp_profit, "kp_activity": kp_activity}, chunksize=5000, force_rebuild=False, batch_id=IMPORT_BATCH, finalize_batch=False)
    for name, ok in results.items():
        print(f"{'OK' if ok else 'FAIL'}: {name}")


In [ ]:
# 品牌3 表格导入 MySQL（按日覆盖 append；info 整表 replace）
# 密码：export MYSQL_PASSWORD='***'
# 若旧表仍是 TEXT/FLOAT、无业务索引，首次请加 force_rebuild=True 重建表结构
with make_exporter() as exporter:
    results = export_brand_tables(exporter, {"zz_account": zz_account, "zz_info": zz_info, "zz_order": zz_order, "zz_exposure": zz_exposure, "zz_product": zz_product}, chunksize=5000, force_rebuild=False, batch_id=IMPORT_BATCH, finalize_batch=True)
    for name, ok in results.items():
        print(f"{'OK' if ok else 'FAIL'}: {name}")
